# CMGAN Training Notebook

Notebook-Version der Trainingslogik aus train.py.

Diese Fassung konzentriert sich auf Datenladen, Modellaufbau, Losses und die Trainingsschleife. LwF ist hier noch nicht enthalten.

In [ ]:
%pip install torchinfo natsort einops

In [ ]:
import csv
import logging
import os

import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
from torchinfo import summary

from data import dataloader
from models import discriminator
from models.generator import TSCNet
from utils import power_compress, power_uncompress

logging.basicConfig(level=logging.INFO)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

In [ ]:
epochs = 120
batch_size = 4
log_interval = 500
decay_epoch = 30
init_lr = 5e-4
cut_len = 16000 * 2
data_dir = "/home/lugra002/speechbrain/data/cmgan_voicebank"
save_model_dir = "./saved_model"
metrics_csv_name = "train_metrics.csv"
loss_weights = [0.1, 0.9, 0.2, 0.05]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = 2

def load_data_notebook(ds_dir, batch_size, num_workers, cut_len):
    torchaudio.set_audio_backend("sox_io")
    train_dir = os.path.join(ds_dir, "train")
    test_dir = os.path.join(ds_dir, "test")

    train_ds = dataloader.DemandDataset(train_dir, cut_len)
    test_ds = dataloader.DemandDataset(test_dir, cut_len)

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
    )

    return train_loader, test_loader

In [ ]:
class Trainer:
    def __init__(self, train_ds, test_ds):
        self.n_fft = 400
        self.hop = 100
        self.train_ds = train_ds
        self.test_ds = test_ds

        self.model = TSCNet(num_channel=64, num_features=self.n_fft // 2 + 1).to(device)
        try:
            summary(self.model, [(1, 2, cut_len // self.hop + 1, int(self.n_fft / 2) + 1)])
        except Exception as exc:
            logging.info("Model summary skipped: %s", exc)

        self.discriminator = discriminator.Discriminator(ndf=16).to(device)
        try:
            summary(
                self.discriminator,
                [
                    (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
                    (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
                ],
            )
        except Exception as exc:
            logging.info("Discriminator summary skipped: %s", exc)

        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=init_lr)
        self.optimizer_disc = torch.optim.AdamW(self.discriminator.parameters(), lr=2 * init_lr)
        self.metrics_csv_path = os.path.join(save_model_dir, metrics_csv_name)

    def forward_generator_step(self, clean, noisy):
        c = torch.sqrt(noisy.size(-1) / torch.sum((noisy**2.0), dim=-1)).unsqueeze(-1)
        noisy = noisy * c
        clean = clean * c

        window = torch.hamming_window(self.n_fft, device=device)
        noisy_spec = torch.stft(noisy, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)
        clean_spec = torch.stft(clean, self.n_fft, self.hop, window=window, onesided=True, return_complex=False)

        noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
        clean_spec = power_compress(clean_spec)
        clean_real = clean_spec[:, 0, :, :].unsqueeze(1)
        clean_imag = clean_spec[:, 1, :, :].unsqueeze(1)

        est_real, est_imag = self.model(noisy_spec)
        est_real = est_real.permute(0, 1, 3, 2)
        est_imag = est_imag.permute(0, 1, 3, 2)
        est_mag = torch.sqrt(est_real**2 + est_imag**2)
        clean_mag = torch.sqrt(clean_real**2 + clean_imag**2)

        est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
        est_audio = torch.istft(
            torch.view_as_complex(est_spec_uncompress.contiguous()),
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )

        return {
            "est_real": est_real,
            "est_imag": est_imag,
            "est_mag": est_mag,
            "clean_real": clean_real,
            "clean_imag": clean_imag,
            "clean_mag": clean_mag,
            "est_audio": est_audio,
        }

    def generator_loss_components(self, generator_outputs):
        predict_fake_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["est_mag"])
        gen_loss_gan = F.mse_loss(predict_fake_metric.flatten(), generator_outputs["one_labels"].float())

        loss_mag = F.mse_loss(generator_outputs["est_mag"], generator_outputs["clean_mag"])
        loss_ri = F.mse_loss(generator_outputs["est_real"], generator_outputs["clean_real"]) + F.mse_loss(
            generator_outputs["est_imag"], generator_outputs["clean_imag"]
        )
        time_loss = torch.mean(torch.abs(generator_outputs["est_audio"] - generator_outputs["clean"]))

        total_loss = (
            loss_weights[0] * loss_ri
            + loss_weights[1] * loss_mag
            + loss_weights[2] * time_loss
            + loss_weights[3] * gen_loss_gan
        )

        return total_loss, {
            "loss_ri": loss_ri.detach().item(),
            "loss_mag": loss_mag.detach().item(),
            "time_loss": time_loss.detach().item(),
            "gen_loss_gan": gen_loss_gan.detach().item(),
        }

    def calculate_generator_loss(self, generator_outputs):
        total_loss, _ = self.generator_loss_components(generator_outputs)
        return total_loss

    def calculate_discriminator_loss(self, generator_outputs):
        length = generator_outputs["est_audio"].size(-1)
        est_audio_list = list(generator_outputs["est_audio"].detach().cpu().numpy())
        clean_audio_list = list(generator_outputs["clean"].detach().cpu().numpy()[:, :length])
        pesq_score = discriminator.batch_pesq(clean_audio_list, est_audio_list)

        if pesq_score is None:
            return None, None

        predict_enhance_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["est_mag"].detach())
        predict_max_metric = self.discriminator(generator_outputs["clean_mag"], generator_outputs["clean_mag"])
        disc_loss = F.mse_loss(predict_max_metric.flatten(), generator_outputs["one_labels"]) + F.mse_loss(
            predict_enhance_metric.flatten(), pesq_score
        )
        return disc_loss, float(pesq_score.detach().mean().item())

    def train_step(self, batch):
        clean = batch[0].to(device)
        noisy = batch[1].to(device)
        one_labels = torch.ones(clean.size(0), device=device)

        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        loss, g_stats = self.generator_loss_components(generator_outputs)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        discrim_loss_metric, pesq_mean = self.calculate_discriminator_loss(generator_outputs)
        if discrim_loss_metric is not None:
            self.optimizer_disc.zero_grad()
            discrim_loss_metric.backward()
            self.optimizer_disc.step()
        else:
            discrim_loss_metric = torch.tensor([0.0], device=device)

        return {
            "gen_loss": loss.item(),
            "disc_loss": discrim_loss_metric.item(),
            "pesq_mean": pesq_mean,
            **g_stats,
        }

    @torch.no_grad()
    def test_step(self, batch):
        clean = batch[0].to(device)
        noisy = batch[1].to(device)
        one_labels = torch.ones(clean.size(0), device=device)

        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        loss, g_stats = self.generator_loss_components(generator_outputs)
        discrim_loss_metric, pesq_mean = self.calculate_discriminator_loss(generator_outputs)
        if discrim_loss_metric is None:
            discrim_loss_metric = torch.tensor([0.0], device=device)

        return {
            "gen_loss": loss.item(),
            "disc_loss": discrim_loss_metric.item(),
            "pesq_mean": pesq_mean,
            **g_stats,
        }

    def _init_epoch_tracker(self):
        return {
            "steps": 0,
            "gen_loss": 0.0,
            "disc_loss": 0.0,
            "loss_ri": 0.0,
            "loss_mag": 0.0,
            "time_loss": 0.0,
            "gen_loss_gan": 0.0,
            "pesq_sum": 0.0,
            "pesq_count": 0,
        }

    def _update_epoch_tracker(self, tracker, step_stats):
        tracker["steps"] += 1
        tracker["gen_loss"] += step_stats["gen_loss"]
        tracker["disc_loss"] += step_stats["disc_loss"]
        tracker["loss_ri"] += step_stats["loss_ri"]
        tracker["loss_mag"] += step_stats["loss_mag"]
        tracker["time_loss"] += step_stats["time_loss"]
        tracker["gen_loss_gan"] += step_stats["gen_loss_gan"]

        if step_stats["pesq_mean"] is not None:
            tracker["pesq_sum"] += step_stats["pesq_mean"]
            tracker["pesq_count"] += 1

    def _finalize_epoch_tracker(self, tracker):
        steps = max(tracker["steps"], 1)
        stats = {
            "gen_loss": tracker["gen_loss"] / steps,
            "disc_loss": tracker["disc_loss"] / steps,
            "loss_ri": tracker["loss_ri"] / steps,
            "loss_mag": tracker["loss_mag"] / steps,
            "time_loss": tracker["time_loss"] / steps,
            "gen_loss_gan": tracker["gen_loss_gan"] / steps,
            "pesq_mean": None,
        }
        if tracker["pesq_count"] > 0:
            stats["pesq_mean"] = tracker["pesq_sum"] / tracker["pesq_count"]
        return stats

    def test(self):
        self.model.eval()
        self.discriminator.eval()

        tracker = self._init_epoch_tracker()
        for batch in self.test_ds:
            step_stats = self.test_step(batch)
            self._update_epoch_tracker(tracker, step_stats)

        stats = self._finalize_epoch_tracker(tracker)
        logging.info(
            "Val | gen: %.4f disc: %.4f ri: %.4f mag: %.4f time: %.4f gan: %.4f pesq: %s",
            stats["gen_loss"],
            stats["disc_loss"],
            stats["loss_ri"],
            stats["loss_mag"],
            stats["time_loss"],
            stats["gen_loss_gan"],
            "n/a" if stats["pesq_mean"] is None else f"{stats['pesq_mean']:.4f}",
        )
        return stats

    def _ensure_metrics_csv(self):
        if os.path.exists(self.metrics_csv_path):
            return

        with open(self.metrics_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    "epoch",
                    "train_gen_loss",
                    "train_disc_loss",
                    "train_loss_ri",
                    "train_loss_mag",
                    "train_time_loss",
                    "train_gen_loss_gan",
                    "train_pesq_mean",
                    "val_gen_loss",
                    "val_disc_loss",
                    "val_loss_ri",
                    "val_loss_mag",
                    "val_time_loss",
                    "val_gen_loss_gan",
                    "val_pesq_mean",
                ]
            )

    def _append_metrics_csv(self, epoch, train_stats, val_stats):
        with open(self.metrics_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    epoch,
                    train_stats["gen_loss"],
                    train_stats["disc_loss"],
                    train_stats["loss_ri"],
                    train_stats["loss_mag"],
                    train_stats["time_loss"],
                    train_stats["gen_loss_gan"],
                    train_stats["pesq_mean"],
                    val_stats["gen_loss"],
                    val_stats["disc_loss"],
                    val_stats["loss_ri"],
                    val_stats["loss_mag"],
                    val_stats["time_loss"],
                    val_stats["gen_loss_gan"],
                    val_stats["pesq_mean"],
                ]
            )

    def train(self):
        scheduler_G = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=decay_epoch, gamma=0.5)
        scheduler_D = torch.optim.lr_scheduler.StepLR(self.optimizer_disc, step_size=decay_epoch, gamma=0.5)
        os.makedirs(save_model_dir, exist_ok=True)
        self._ensure_metrics_csv()

        for epoch in range(epochs):
            self.model.train()
            self.discriminator.train()

            train_tracker = self._init_epoch_tracker()
            for idx, batch in enumerate(self.train_ds):
                step = idx + 1
                step_stats = self.train_step(batch)
                self._update_epoch_tracker(train_tracker, step_stats)

                if step % log_interval == 0:
                    logging.info(
                        "Epoch %s Step %s | gen: %.4f disc: %.4f ri: %.4f mag: %.4f time: %.4f gan: %.4f pesq: %s",
                        epoch,
                        step,
                        step_stats["gen_loss"],
                        step_stats["disc_loss"],
                        step_stats["loss_ri"],
                        step_stats["loss_mag"],
                        step_stats["time_loss"],
                        step_stats["gen_loss_gan"],
                        "n/a" if step_stats["pesq_mean"] is None else f"{step_stats['pesq_mean']:.4f}",
                    )

            train_stats = self._finalize_epoch_tracker(train_tracker)
            val_stats = self.test()

            logging.info(
                "Epoch %s Summary | train_gen: %.4f train_disc: %.4f val_gen: %.4f val_disc: %.4f",
                epoch,
                train_stats["gen_loss"],
                train_stats["disc_loss"],
                val_stats["gen_loss"],
                val_stats["disc_loss"],
            )

            self._append_metrics_csv(epoch, train_stats, val_stats)

            path = os.path.join(save_model_dir, "CMGAN_epoch_" + str(epoch) + "_" + str(val_stats["gen_loss"])[:5])
            torch.save(self.model.state_dict(), path)
            scheduler_G.step()
            scheduler_D.step()

In [ ]:
train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
trainer = Trainer(train_ds, test_ds)
trainer

In [ ]:
# Startet das Training manuell, wenn die Datenpfade gesetzt sind.
trainer.train()